In [1]:
import os

In [2]:
%pwd

'f:\\Chicken-Disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'f:\\Chicken-Disease-Classification'

In [9]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    container_name: str
    blob_name: str
    local_data_file: Path
    unzip_dir: Path

In [10]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            container_name=config.container_name,
            blob_name=config.blob_name,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [12]:
import os
import zipfile

from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import AzureError

from cnnClassifier import logger


load_dotenv()

c:\Users\feroz\miniconda3\envs\cnncls\lib\site-packages\azure\storage\blob\_encryption.py:22: CryptographyDeprecationWarning: Python 3.8 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.8.
  from cryptography.hazmat.backends import default_backend


True

In [15]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_file(self):
        """
        Downloads data.zip from Azure Blob Storage.
        """

        if os.path.exists(self.config.local_data_file):
            logger.info(
                f"File already exists: {self.config.local_data_file}"
            )
            return

        connection_string = os.getenv(
            "AZURE_STORAGE_CONNECTION_STRING"
        )

        if not connection_string:
            raise ValueError(
                "AZURE_STORAGE_CONNECTION_STRING not found in .env"
            )

        try:
            blob_service_client = (
                BlobServiceClient.from_connection_string(
                    connection_string
                )
            )

            blob_client = blob_service_client.get_blob_client(
                container=self.config.container_name,
                blob=self.config.blob_name
            )

            os.makedirs(
                os.path.dirname(self.config.local_data_file),
                exist_ok=True
            )

            logger.info(
                f"Downloading {self.config.blob_name} "
                f"from Azure container "
                f"{self.config.container_name}"
            )

            with open(
                self.config.local_data_file,
                "wb"
            ) as file:

                download_stream = blob_client.download_blob(
                    max_concurrency=4
                )

                download_stream.readinto(file)

            logger.info("Azure Blob download completed.")

        except AzureError as e:
            logger.error(f"Azure download failed: {e}")
            raise


    def extract_zip_file(self):
        """
        Extracts downloaded zip file.
        """

        unzip_path = self.config.unzip_dir

        os.makedirs(
            unzip_path,
            exist_ok=True
        )

        with zipfile.ZipFile(
            self.config.local_data_file,
            "r"
        ) as zip_ref:

            zip_ref.extractall(unzip_path)

        logger.info(
            f"Dataset extracted to: {unzip_path}"
        )

In [16]:
try:
    config = ConfigurationManager()

    data_ingestion_config = (
        config.get_data_ingestion_config()
    )

    data_ingestion = DataIngestion(
        config=data_ingestion_config
    )

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e 

[2026-08-24 22:00:08,839: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-24 22:00:08,841: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-24 22:00:08,842: INFO: common: created directory at: artifacts]
[2026-08-24 22:00:08,843: INFO: common: created directory at: artifacts/data_ingestion]
[2026-08-24 22:00:08,844: INFO: 1874351221: Downloading data.zip from Azure container chicken-data-2025]
[2026-08-24 22:00:08,853: INFO: _universal: Request URL: 'https://chickenmlstorage2026.blob.core.windows.net/chicken-data-2025/data.zip'
Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.26.0 Python/3.8.20 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'e1643843-9fd4-11f1-b3a3-18c04db194a5'
    'Authorization': 'REDACTED'
No body was attached to the request]
[2026-08-24 22:00:09,439: INF